# Dynamic Factor Allocation — Stage 2 (Transforms) & Stage 3 (Feature Engineering)

Reproduction of the data pipeline for **Shu & Mulvey (2024), "Dynamic Factor Allocation Leveraging Regime-Switching Signals"**.

This notebook builds **only** the transforms and the per-factor feature tables (paper Exhibit 3). It does **NOT** build the regime-switching model (SJM).

**Input:** `merged_data_dynamic_factor.parquet` (daily, 1999-01-01 → 2026-06-15, 7162 rows).

**Outputs:**
- `features/{value,size,momentum,quality,lowvol,growth}.parquet` — one ~17-column feature table per factor.
- `active_returns.parquet` — the 6 active-return series.
- `cumulative_active_returns.png` — Exhibit-2-style plot.

All window lengths are in **trading days**. EWMA features use `ewm(span=W)`; RSI/%K use rolling windows.

In [ ]:
# ----------------------------------------------------------------------
# 0. Imports & configuration
# ----------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Input file (note: actual filename has the _dynamic_factor suffix)
INPUT_FILE = "merged_data_dynamic_factor.parquet"

FEATURES_DIR = Path("features")
FEATURES_DIR.mkdir(exist_ok=True)

# The 6 factors (active returns will be computed for each).
FACTORS = ["value", "size", "momentum", "quality", "lowvol", "growth"]

# Feature window lengths (trading days).
SPANS = [8, 21, 63]          # used for EWMA / RSI / %K / MACD
DD_WINDOW = 21               # downside-deviation window
BETA_SPAN = 21               # EWM beta span
MACRO_SPAN = 21              # shared market-environment EWMA span

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

In [ ]:
# ----------------------------------------------------------------------
# 1. Load merged data
# ----------------------------------------------------------------------
df = pd.read_parquet(INPUT_FILE)
df = df.sort_index()                       # ensure chronological order
df.index.name = "date"

print("Loaded:", df.shape)
print("Date range:", df.index.min().date(), "->", df.index.max().date())
print("Columns:", list(df.columns))
print("\nNaN counts (raw):")
print(df.isna().sum())
df.head()

## Stage 2 — Transforms

1. **Active returns**: `active_f = ret_f - ret_mkt` for each factor.
2. **Macro transforms**:
   - `vix_ld   = log(vix).diff()`
   - `dgs2_d   = dgs2.diff()`
   - `slope    = dgs10 - dgs2`
   - `slope_d  = slope.diff()`

`dgs3mo` is the risk-free rate for later excess returns — it is **not** a feature here.

In [ ]:
# ----------------------------------------------------------------------
# 2. Stage 2 transforms
# ----------------------------------------------------------------------
# --- 2a. Active returns: factor minus market ---
active = pd.DataFrame(index=df.index)
for f in FACTORS:
    active[f] = df[f] - df["mkt"]

# NOTE on missing values:
#   growth (~250) and momentum (~22) have scattered calendar-mismatch NaNs, which
#   propagate into active returns and would otherwise cascade through cumprod /
#   rolling windows and blow a hole far larger than the intended ~63-row warmup.
#   We treat a missing daily active return as a 0% active move for that day
#   (no information => no active drift). This keeps the cumulative price series
#   continuous. The fill is applied ONLY for feature computation.
active_filled = active.fillna(0.0)

# Market return, also gap-filled with 0 for the EWM beta / shared macro feature.
mkt = df["mkt"].fillna(0.0)

# --- 2b. Macro transforms ---
vix_ld  = np.log(df["vix"]).diff()          # log-difference of VIX level
dgs2_d  = df["dgs2"].diff()                 # first difference of 2Y yield
slope   = df["dgs10"] - df["dgs2"]          # 10Y - 2Y term-structure slope
slope_d = slope.diff()                      # first difference of the slope

print("Active returns (head):")
print(active.head())
print("\nActive-return NaN counts (pre-fill):")
print(active.isna().sum())

# Persist the (un-filled) active returns for inspection.
active.to_parquet("active_returns.parquet")
print("\nSaved active_returns.parquet")

## Stage 3 — Feature engineering (Exhibit 3)

For each factor, computed on **that factor's active returns**:

| Group | Features |
|---|---|
| EWMA of active return | `r_ewma_8`, `r_ewma_21`, `r_ewma_63` |
| RSI of cum. active-return price | `rsi_8`, `rsi_21`, `rsi_63` |
| Stochastic %K of cum. price | `k_8`, `k_21`, `k_63` |
| MACD of cum. price | `macd_8_21`, `macd_21_63` |
| Downside deviation | `log_dd_21` |
| EWM active market beta | `beta_21` |
| Market env. (shared) | `mkt_ewma_21`, `vix_ewma_21`, `dgs2_ewma_21`, `slope_ewma_21` |

The "cumulative active-return price series" is `price = (1 + active_return).cumprod()`.

In [ ]:
# ----------------------------------------------------------------------
# 3. Feature helper functions (standard technical-indicator definitions)
# ----------------------------------------------------------------------
def rsi(price: pd.Series, window: int) -> pd.Series:
    """Relative Strength Index on a price series using simple rolling
    average gain/loss over `window`. Returns values in [0, 100]."""
    delta = price.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / avg_loss
    out = 100 - 100 / (1 + rs)
    # If avg_loss == 0 over the window (no down days) RSI is defined as 100.
    out = out.where(avg_loss != 0, 100.0)
    # Keep the warmup NaNs (first `window` rows) as NaN.
    out[avg_gain.isna()] = np.nan
    return out


def stoch_k(price: pd.Series, window: int) -> pd.Series:
    """Stochastic Oscillator %K = (price - min) / (max - min) * 100
    over a rolling `window`. Returns values in [0, 100]."""
    lo = price.rolling(window).min()
    hi = price.rolling(window).max()
    rng = hi - lo
    k = (price - lo) / rng * 100
    # Flat window (hi == lo) -> undefined ratio; convention = 50 (mid-range).
    k = k.where(rng != 0, 50.0)
    return k


def downside_deviation(ret: pd.Series, window: int) -> pd.Series:
    """Rolling downside deviation = sqrt(mean of squared NEGATIVE returns)
    over `window`."""
    neg_sq = ret.clip(upper=0) ** 2          # squared returns; 0 where ret >= 0
    return np.sqrt(neg_sq.rolling(window).mean())


def ewm_beta(ret: pd.Series, market: pd.Series, span: int) -> pd.Series:
    """EWM market beta = EWMcov(ret, market) / EWMvar(market)."""
    cov = ret.ewm(span=span).cov(market)
    var = market.ewm(span=span).var()
    return cov / var

In [ ]:
# ----------------------------------------------------------------------
# 4. Shared market-environment features (identical across all 6 factors)
# ----------------------------------------------------------------------
macro = pd.DataFrame(index=df.index)
macro["mkt_ewma_21"]   = mkt.ewm(span=MACRO_SPAN).mean()
macro["vix_ewma_21"]   = vix_ld.ewm(span=MACRO_SPAN).mean()
macro["dgs2_ewma_21"]  = dgs2_d.ewm(span=MACRO_SPAN).mean()
macro["slope_ewma_21"] = slope_d.ewm(span=MACRO_SPAN).mean()
macro.tail()

In [ ]:
# ----------------------------------------------------------------------
# 5. Build per-factor feature tables
# ----------------------------------------------------------------------
def build_features(factor: str) -> pd.DataFrame:
    """Construct the full feature table for one factor's active returns."""
    r = active_filled[factor]                      # active returns (gap-filled)
    price = (1 + r).cumprod()                       # cumulative active-return price

    feat = pd.DataFrame(index=df.index)

    # --- EWMA of the active return ---
    for w in SPANS:
        feat[f"r_ewma_{w}"] = r.ewm(span=w).mean()

    # --- RSI of the cumulative-price series ---
    for w in SPANS:
        feat[f"rsi_{w}"] = rsi(price, w)

    # --- Stochastic %K of the cumulative-price series ---
    for w in SPANS:
        feat[f"k_{w}"] = stoch_k(price, w)

    # --- MACD (difference of two EWMAs of the price series) ---
    ema = {w: price.ewm(span=w).mean() for w in SPANS}
    feat["macd_8_21"]  = ema[8]  - ema[21]
    feat["macd_21_63"] = ema[21] - ema[63]

    # --- Downside deviation (log) ---
    dd = downside_deviation(r, DD_WINDOW)
    # log(0) -> -inf if a 21-day window has no negative active returns (rare);
    # map those to NaN so the column stays finite.
    feat["log_dd_21"] = np.log(dd.replace(0.0, np.nan))

    # --- EWM active market beta ---
    feat["beta_21"] = ewm_beta(r, mkt, BETA_SPAN)

    # --- Shared market-environment features ---
    feat = feat.join(macro)

    # --- Drop the leading warmup block (first fully-valid row onward) ---
    # The longest window is 63, so ~63 leading rows are NaN.
    first_valid = feat.dropna().index[0]
    feat = feat.loc[first_valid:]
    feat.index.name = "date"
    return feat


feature_tables = {f: build_features(f) for f in FACTORS}
print("Built feature tables for:", list(feature_tables.keys()))
print("Columns per table (%d):" % feature_tables["value"].shape[1])
print(list(feature_tables["value"].columns))

In [ ]:
# ----------------------------------------------------------------------
# 6. Save feature tables
# ----------------------------------------------------------------------
for f, tbl in feature_tables.items():
    path = FEATURES_DIR / f"{f}.parquet"
    tbl.to_parquet(path)
    print(f"Saved {path}  shape={tbl.shape}")

In [ ]:
# ----------------------------------------------------------------------
# 7. Per-factor diagnostics: shape, date range, columns, NaN counts, describe
# ----------------------------------------------------------------------
for f in FACTORS:
    tbl = feature_tables[f]
    print("=" * 90)
    print(f"FACTOR: {f.upper()}")
    print("-" * 90)
    print("shape      :", tbl.shape)
    print("date range :", tbl.index.min().date(), "->", tbl.index.max().date())
    print("columns    :", list(tbl.columns))
    print("\nNaN counts:")
    print(tbl.isna().sum())
    print("\n.describe():")
    print(tbl.describe().T)
    print()

In [ ]:
# ----------------------------------------------------------------------
# 8. Plot cumulative active returns (Exhibit 2 sanity check)
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 7))
for f in FACTORS:
    cum = (1 + active_filled[f]).cumprod()
    ax.plot(cum.index, cum.values, label=f, linewidth=1.2)

ax.set_title("Cumulative Active Returns by Factor  (factor return - market return)")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative active growth of $1")
ax.axhline(1.0, color="grey", linewidth=0.8, linestyle="--")
ax.legend(ncol=3)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("cumulative_active_returns.png", dpi=150)
print("Saved cumulative_active_returns.png")
plt.show()

---
**Done — Stage 2 & Stage 3 only.** The regime-switching model (SJM) is intentionally not built here.